# 02 · ข้อ 1 วันธรรมดา vs วันหยุดสุดสัปดาห์
ผู้รับผิดชอบ: สมาชิก A · ต้องรัน `01_cleaning.ipynb` (หรือ `python src/run_all.py`) ก่อน

**คำถาม:** ปริมาณการใช้บริการขนส่งแตกต่างกันอย่างไรระหว่างวันธรรมดาและวันหยุดสุดสัปดาห์ ต่างกันอย่างไรในแต่ละวันของสัปดาห์ และรูปแบบนี้เปลี่ยนไปในแต่ละเดือนหรือไม่

โค้ดทั้งหมดอยู่ใน `src/q1_weekday.py` notebook นี้มีหน้าที่**เรียกฟังก์ชันและแสดงผล**พร้อมคำอธิบาย
ฟังก์ชันที่ยังเป็น TODO ให้เขียนใน `src/q1_weekday.py` → ตรวจด้วย `pytest tests/test_q1_weekday.py` → ลบ `#` หน้า cell ที่เกี่ยวข้อง

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import prep
import features
import plots

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
_ = plots.setup_style()
import q1_weekday

df = prep.load_clean()
print(df.shape)

(18705, 18)


## 1. Feature Selection

| Feature ต้นฉบับ | เหตุผล |
|---|---|
| `date` | สร้างวันในสัปดาห์และเดือน |
| `service`, `purpose`, `mode` | เปรียบเทียบรูปแบบระหว่างบริการ/กลุ่มการเดินทาง |
| `value` | จำนวนผู้ใช้บริการต่อวัน |
| `unit`, `exclude_reason`, `is_aggregate`, `missing_flag`, `zero_long_run` | ใช้กรอง |

## 2. Data Preparation

In [2]:
# ปี 2568 · หน่วย "คน" · ตัดค่าว่างและศูนย์ช่วงหยุดรายงาน (ดู docstring ของ select_rows)
rows = q1_weekday.select_rows(df)
print(rows.shape, rows["service"].nunique(), "services")
print("แถวที่ตัดเพราะค่าว่าง/ศูนย์ช่วงยาว:", len(prep.passenger_rows(df).query("year_be == 2568")) - len(rows))

(13664, 18) 38 services
แถวที่ตัดเพราะค่าว่าง/ศูนย์ช่วงยาว: 206


## 3. Feature Engineering

| Feature ใหม่ | สูตร / เงื่อนไข | ฟังก์ชัน |
|---|---|---|
| `day_of_week`, `month`, `is_weekend` | `dayofweek`, `month`, `dayofweek ∈ {5,6}` | `features.add_calendar_features` ✅ |
| `dow_index` | `value / mean(value ของบริการทั้งปี) × 100` | `q1_weekday.build_daily_features` (TODO) |
| `weekend_ratio` | `mean(weekend) / mean(weekday)` ต่อบริการ × เดือน | `q1_weekday.weekend_ratio_table` (TODO) |
| `pct_diff_weekend` | `(weekend_ratio − 1) × 100` | `q1_weekday.weekend_ratio_table` (TODO) |

In [3]:
# daily = q1_weekday.build_daily_features(rows)
# ratio = q1_weekday.weekend_ratio_table(daily)
# display(daily.head(), ratio.describe())

## 4. Visualization
กราฟที่วางไว้: ① `plot_dow_index` ② `plot_month_dow_heatmap`

In [4]:
# fig = q1_weekday.plot_dow_index(daily); plots.save_fig(fig, "q1_dow_index"); plt.show()
# fig = q1_weekday.plot_month_dow_heatmap(daily); plots.save_fig(fig, "q1_month_dow_heatmap"); plt.show()

**การตีความ:** _(เขียนจากตัวเลขจริงในกราฟ)_

## 5. Analysis & Findings
- คำตอบของคำถาม:
- ข้อค้นพบสำคัญ:
- ข้อจำกัดของข้อมูล:

## 6. บันทึก Prepared Dataset

In [5]:
# daily.to_csv(prep.PREPARED_DIR / q1_weekday.OUTPUT_FILE, index=False, encoding="utf-8-sig")